# W&B Run Download And Local Plots

This notebook downloads selected W&B run histories, caches them locally, and plots training/evaluation curves without depending on the W&B UI.

Before running it, make sure you are logged in:

```bash
wandb login
```

The downloaded cache is written to `Topology_Task/outputs/wandb_cache/` and figures are written to `Topology_Task/outputs/wandb_figures/`.

In [2]:
from pathlib import Path
import json
import os
import re
import shutil
import time

import numpy as np
import pandas as pd

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError as exc:
    raise ImportError("Install plotly first, for example: pip install plotly") from exc

try:
    import wandb
except ImportError:
    wandb = None

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)


## Configuration

Edit this cell to select runs. The default regex targets the entropy-decay `s0/s1/s2` seed sweep.

In [3]:
ENTITY = os.getenv("WANDB_ENTITY", "corentin-plumet-epfl")
PROJECT = os.getenv("WANDB_PROJECT", "Grid2Op")

# None means every W&B run in the project. Use a regex to focus on a subset.
ENTROPY_DECAY_SEED_SWEEP_REGEX = (
    r"^noval20_mlp_(?:a1_entropy_decay|a3_entropy_decay|a3_logit_decay|a4_entropy_decay|p999_decay)"
    r"(?:_s[0-2])?_(?:det|stoch)$"
)
NO_ENTROPY_DECAY_SEED_SWEEP_REGEX = (
    r"^noval20_mlp_(?:a1_no_entropy_decay|a3_no_entropy_decay|a3_logit_decay_no_entropy_decay|a4_no_entropy_decay)_"
    r"s[0-2]_(?:det|stoch)$"
)
RUN_NAME_REGEX = None
EXCLUDE_RUN_NAME_REGEX = None
RUN_STATES = None  # None includes finished, crashed, and killed cached histories.
MAX_RUNS = None    # None means all matching runs

# Full history is downloaded from W&B history artifacts: run-<run_id>-history:<version>.
CACHE_MODE = "full"
FORCE_REFRESH = False
USE_LOCAL_CACHE_ONLY = True  # False downloads missing histories from W&B; True only reads local cache.
WRITE_FULL_HISTORY_CSV = True
SKIP_FAILED_DOWNLOADS = True  # Continue when active runs do not have history artifacts yet.
WANDB_API_TIMEOUT = 300
HISTORY_ARTIFACT_TYPE = "wandb-history"
HISTORY_ARTIFACT_VERSION = "latest"
HISTORY_ARTIFACT_FALLBACK_VERSIONS = ["v0", "v1", "v2", "v3", "v4", "v5"]

METRICS = [
    "charts/episodic_survival",
    "test/charts/episodic_survival",
    "test/episodic_survival",
    "validation/episodic_survival",
    "train_eval/charts/episodic_survival",
    "train_eval/episodic_survival",
    "train/entropy_coef",
    "train/action0_logit_bonus",
    "train/entropy_agent_0",
    "train/entropy_agent_1",
    "train/entropy_agent_2",
    "train/frac_action_0_agent_0",
    "train/frac_action_0_agent_1",
    "train/frac_action_0_agent_2",
    "train/approx_kl_agent_0",
    "train/approx_kl_agent_1",
    "train/approx_kl_agent_2",
    "train/lr_actor",
    "train/lr_critic",
]

cwd = Path.cwd().resolve()
if (cwd / "main.py").is_file():
    TASK_DIR = cwd
elif (cwd / "Topology_Task" / "main.py").is_file():
    TASK_DIR = cwd / "Topology_Task"
elif (cwd.parent / "main.py").is_file():
    TASK_DIR = cwd.parent
else:
    raise RuntimeError("Could not locate Topology_Task/main.py from the current working directory.")

CACHE_DIR = TASK_DIR / "outputs" / "wandb_cache"
FULL_CACHE_DIR = CACHE_DIR / "full_history"
METRIC_CACHE_DIR = CACHE_DIR / "metric_history"
FIG_DIR = TASK_DIR / "outputs" / "wandb_figures"
for directory in [CACHE_DIR, FULL_CACHE_DIR, METRIC_CACHE_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = CACHE_DIR / "selected_runs_manifest.csv"
CACHE_INDEX_PATH = CACHE_DIR / "full_history_cache_index.csv"
FAILED_HISTORY_DOWNLOADS_PATH = CACHE_DIR / "failed_history_downloads.csv"

print(f"Project: {ENTITY}/{PROJECT}")
print(f"Task dir: {TASK_DIR}")
print(f"Cache mode: {CACHE_MODE}")
print(f"Cache dir: {CACHE_DIR}")
print(f"Local-only mode: {USE_LOCAL_CACHE_ONLY}")


Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True


## Fetch Run List

In [4]:
name_re = re.compile(RUN_NAME_REGEX) if RUN_NAME_REGEX else None
exclude_name_re = re.compile(EXCLUDE_RUN_NAME_REGEX) if EXCLUDE_RUN_NAME_REGEX else None


def run_name_matches(name, exp_tag=""):
    if exclude_name_re is not None and (
        exclude_name_re.search(str(name)) or exclude_name_re.search(str(exp_tag))
    ):
        return False
    if name_re is None:
        return True
    return bool(name_re.search(str(name)) or name_re.search(str(exp_tag)))


def _metadata_json_paths():
    return sorted(FULL_CACHE_DIR.glob("*/metadata.json"))


def _cache_file_from_metadata(meta, meta_path, key, filename):
    raw_path = meta.get(key)
    if raw_path:
        path = Path(raw_path)
        if path.exists():
            return path
    fallback = meta_path.parent / filename
    return fallback if fallback.exists() else None


def cached_runs_from_full_history():
    rows = []
    for meta_path in _metadata_json_paths():
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError as exc:
            print(f"Skipping invalid metadata {meta_path}: {exc}")
            continue

        name = meta.get("name") or meta.get("run_name") or meta_path.parent.name.split("__", 1)[0]
        run_id = meta.get("id") or meta.get("run_id") or meta_path.parent.name.split("__")[-1]
        exp_tag = meta.get("exp_tag") or name
        parquet_path = _cache_file_from_metadata(meta, meta_path, "history_parquet", "history.parquet")
        csv_path = _cache_file_from_metadata(meta, meta_path, "history_csv", "history.csv.gz")
        if parquet_path is None and csv_path is None:
            print(f"Skipping {name}: no history.parquet or history.csv.gz found")
            continue

        rows.append({
            "name": name,
            "id": run_id,
            "state": meta.get("state"),
            "created_at": meta.get("created_at"),
            "exp_tag": exp_tag,
            "actor_encoder": meta.get("actor_encoder"),
            "critic_encoder": meta.get("critic_encoder"),
            "gnn_type": meta.get("gnn_type"),
            "deterministic_eval": meta.get("deterministic_eval"),
            "n_envs": meta.get("n_envs"),
            "n_steps": meta.get("n_steps"),
            "entropy_coef": meta.get("entropy_coef"),
            "entropy_coef_final": meta.get("entropy_coef_final"),
            "init_do_nothing_prob": meta.get("init_do_nothing_prob"),
            "artifact": meta.get("artifact_resolved") or meta.get("artifact_requested"),
            "history_parquet": str(parquet_path) if parquet_path else None,
            "history_csv": str(csv_path) if csv_path else None,
            "rows": meta.get("rows"),
            "columns": meta.get("columns"),
        })
    return pd.DataFrame(rows)


if USE_LOCAL_CACHE_ONLY:
    runs_df = cached_runs_from_full_history()
    if runs_df.empty:
        raise FileNotFoundError(
            f"No cached histories found under {FULL_CACHE_DIR}. Download finished runs first."
        )
    runs_df = runs_df[runs_df.apply(lambda row: run_name_matches(row.get("name"), row.get("exp_tag", "")), axis=1)]
    if RUN_STATES is not None and "state" in runs_df.columns:
        runs_df = runs_df[runs_df["state"].isin(RUN_STATES)]
    runs_df = runs_df.sort_values(["name", "id"]).reset_index(drop=True)
    if MAX_RUNS is not None:
        runs_df = runs_df.head(MAX_RUNS)
    selected_runs = []
    all_runs = []
    print(f"Selected {len(runs_df)} cached runs from {FULL_CACHE_DIR}")
    if "state" in runs_df.columns:
        print(runs_df["state"].value_counts(dropna=False).to_string())
else:
    if wandb is None:
        raise ImportError("Install wandb first, for example: pip install wandb")
    api = wandb.Api(timeout=WANDB_API_TIMEOUT)
    all_runs = list(api.runs(f"{ENTITY}/{PROJECT}"))

    def run_matches(run):
        if RUN_STATES is not None and run.state not in RUN_STATES:
            return False
        exp_tag = str(run.config.get("exp_tag", ""))
        return run_name_matches(run.name, exp_tag)

    selected_runs = [run for run in all_runs if run_matches(run)]
    selected_runs = sorted(selected_runs, key=lambda run: getattr(run, "created_at", "") or "")
    if MAX_RUNS is not None:
        selected_runs = selected_runs[:MAX_RUNS]

    summary_rows = []
    for run in selected_runs:
        summary = dict(run.summary)
        config = dict(run.config)
        summary_rows.append({
            "name": run.name,
            "id": run.id,
            "state": run.state,
            "created_at": getattr(run, "created_at", None),
            "exp_tag": config.get("exp_tag"),
            "actor_encoder": config.get("actor_encoder"),
            "critic_encoder": config.get("critic_encoder"),
            "gnn_type": config.get("gnn_type"),
            "deterministic_eval": config.get("deterministic_eval"),
            "n_envs": config.get("n_envs"),
            "n_steps": config.get("n_steps"),
            "entropy_coef": config.get("entropy_coef"),
            "entropy_coef_final": config.get("entropy_coef_final"),
            "init_do_nothing_prob": config.get("init_do_nothing_prob"),
            "global_step": summary.get("charts/global_step"),
            "test_survival": summary.get("test/charts/episodic_survival", summary.get("test/episodic_survival")),
            "train_eval_survival": summary.get("train_eval/charts/episodic_survival", summary.get("train_eval/episodic_survival")),
            "legacy_survival": summary.get("charts/episodic_survival"),
        })

    runs_df = pd.DataFrame(summary_rows)
    runs_df.to_csv(MANIFEST_PATH, index=False)
    print(f"Selected {len(selected_runs)} / {len(all_runs)} runs")
    print(f"Saved manifest: {MANIFEST_PATH}")

runs_df


Selected 126 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
finished    72
crashed     49
killed       5


,name,id,state,created_at,exp_tag,actor_encoder,critic_encoder,gnn_type,deterministic_eval,n_envs,n_steps,entropy_coef,entropy_coef_final,init_do_nothing_prob,artifact,history_parquet,history_csv,rows,columns
0,MAPPO_baseline_nejjar,MAPPO_bus14_T_0_0__I__1779543882_26350,finished,None,MAPPO_baseline_nejjar,None,None,None,None,None,None,None,None,None,corentin-plumet-epfl/Grid2Op/run-MAPPO_bus14_T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,625,30
1,a0_known_good_det,MAPPO_bus14_T_0_0__I__1779591426_12887,finished,None,a0_known_good_det,None,None,None,None,None,None,None,None,None,corentin-plumet-epfl/Grid2Op/run-MAPPO_bus14_T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,375,30
2,a0_known_good_sto,MAPPO_bus14_T_0_0__I__1779592809_45825,finished,None,a0_known_good_sto,None,None,None,None,None,None,None,None,None,corentin-plumet-epfl/Grid2Op/run-MAPPO_bus14_T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,375,30
3,a0_stoch_20env_mb20_ep15_kl004,MAPPO_bus14_T_0_0__I__1779682811_401,finished,None,a0_stoch_20env_mb20_ep15_kl004,None,None,None,None,None,None,None,None,None,corentin-plumet-epfl/Grid2Op/run-MAPPO_bus14_T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,375,32
4,a1_20env_entropy_to0_stoch,MAPPO_bus14_T_0_0__I__1779675819_23482,finished,None,a1_20env_entropy_to0_stoch,None,None,None,None,None,None,None,None,None,corentin-plumet-epfl/Grid2Op/run-MAPPO_bus14_T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,375,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,split_mlp_a3_logit_decay_det_s0,MAPPO_bus14_T_0_0__I__1779737554_31819,killed,None,split_mlp_a3_logit_decay_det_s0,None,None,None,None,None,None,None,None,None,corentin-plumet-epfl/Grid2Op/run-MAPPO_bus14_T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,328,37
122,split_mlp_a3_logit_decay_stoch_s0,MAPPO_bus14_T_0_0__I__1779738129_44015,finished,None,split_mlp_a3_logit_decay_stoch_s0,None,None,None,None,None,None,None,None,None,corentin-plumet-epfl/Grid2Op/run-MAPPO_bus14_T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,375,37
123,split_mlp_a3_stoch_s0,MAPPO_bus14_T_0_0__I__1779727695_29348,finished,None,split_mlp_a3_stoch_s0,None,None,None,None,None,None,None,None,None,corentin-plumet-epfl/Grid2Op/run-MAPPO_bus14_T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,375,37
124,split_mlp_p999_decay_det_s0,MAPPO_bus14_T_0_0__I__1779729525_26173,killed,None,split_mlp_p999_decay_det_s0,None,None,None,None,None,None,None,None,None,corentin-plumet-epfl/Grid2Op/run-MAPPO_bus14_T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,345,37


## Download And Cache Histories

The full-history cache uses W&B history artifacts instead of `scan_history`. For each selected run, it downloads:

```text
{ENTITY}/{PROJECT}/run-<run_id>-history:latest
```

Each run is stored neatly under `outputs/wandb_cache/full_history/<run_name>__<run_id>/` with the artifact `history.parquet`, optional `history.csv.gz`, and `metadata.json`.

After the cache exists, set `USE_LOCAL_CACHE_ONLY = True` to plot without calling W&B.


In [5]:
def safe_name(text):
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-")
    return text or "run"


def run_cache_dir(run_name, run_id):
    return FULL_CACHE_DIR / f"{safe_name(run_name)}__{run_id}"


def full_parquet_path(run_name, run_id):
    return run_cache_dir(run_name, run_id) / "history.parquet"


def full_csv_path(run_name, run_id):
    return run_cache_dir(run_name, run_id) / "history.csv.gz"


def metadata_path(run_name, run_id):
    return run_cache_dir(run_name, run_id) / "metadata.json"


def artifact_path_for_run(run_id, version=None):
    version = version or HISTORY_ARTIFACT_VERSION
    return f"{ENTITY}/{PROJECT}/run-{run_id}-history:{version}"


def artifact_versions_to_try():
    versions = [HISTORY_ARTIFACT_VERSION] + list(HISTORY_ARTIFACT_FALLBACK_VERSIONS)
    deduped = []
    for version in versions:
        if version and version not in deduped:
            deduped.append(version)
    return deduped


def _progress(prefix, idx, total, name):
    return f"[{idx:>2}/{total}] {prefix}: {name}"


def _row_get(row, key, default=None):
    if isinstance(row, dict):
        return row.get(key, default)
    return getattr(row, key, default)


def _read_history_table(parquet_path, csv_path=None):
    parquet_path = Path(parquet_path) if parquet_path is not None else None
    csv_path = Path(csv_path) if csv_path is not None else None
    if parquet_path is not None and parquet_path.exists():
        try:
            history = pd.read_parquet(parquet_path)
            history.attrs["source_path"] = str(parquet_path)
            return history
        except ImportError as exc:
            if csv_path is not None and csv_path.exists():
                print(f"    parquet reader unavailable; loading CSV cache instead: {csv_path}", flush=True)
                history = pd.read_csv(csv_path)
                history.attrs["source_path"] = str(csv_path)
                return history
            raise ImportError(
                "Reading W&B history parquet requires pyarrow or fastparquet. "
                "Install one of them, for example: pip install pyarrow"
            ) from exc
    if csv_path is not None and csv_path.exists():
        history = pd.read_csv(csv_path)
        history.attrs["source_path"] = str(csv_path)
        return history
    raise FileNotFoundError(f"Missing cached history file: parquet={parquet_path}, csv={csv_path}")


def _ensure_run_columns(history, run_name, run_id):
    history = history.copy()
    if "run_name" not in history.columns:
        history.insert(0, "run_name", run_name)
    else:
        history["run_name"] = history["run_name"].fillna(run_name)
    if "run_id" not in history.columns:
        history.insert(1, "run_id", run_id)
    else:
        history["run_id"] = history["run_id"].fillna(run_id)
    return history


def _find_downloaded_history(download_dir):
    download_dir = Path(download_dir)
    direct = download_dir / "0000.parquet"
    if direct.exists():
        return direct
    matches = sorted(download_dir.rglob("0000.parquet"))
    if not matches:
        raise FileNotFoundError(f"Could not find 0000.parquet under {download_dir}")
    return matches[0]


wandb_download_run = None


def _get_history_artifact(run_id):
    errors = []
    for version in artifact_versions_to_try():
        artifact_ref = artifact_path_for_run(run_id, version=version)
        try:
            artifact = api.artifact(artifact_ref, type=HISTORY_ARTIFACT_TYPE)
            if errors:
                print(f"    resolved history artifact with {artifact_ref}", flush=True)
            return artifact, artifact_ref
        except Exception as api_exc:
            errors.append((artifact_ref, api_exc))
            print(
                f"    api.artifact could not load {artifact_ref}: {type(api_exc).__name__}: {api_exc}",
                flush=True,
            )

    # Fallback to the method recommended in W&B examples.
    global wandb_download_run
    if wandb_download_run is None:
        wandb_download_run = wandb.init(
            project=PROJECT,
            entity=ENTITY,
            job_type="download-history-artifacts",
            name="download_history_artifacts",
            reinit=True,
        )

    for version in artifact_versions_to_try():
        artifact_ref = artifact_path_for_run(run_id, version=version)
        try:
            artifact = wandb_download_run.use_artifact(artifact_ref, type=HISTORY_ARTIFACT_TYPE)
            print(f"    resolved history artifact with {artifact_ref}", flush=True)
            return artifact, artifact_ref
        except Exception as use_exc:
            errors.append((artifact_ref, use_exc))
            print(
                f"    use_artifact could not load {artifact_ref}: {type(use_exc).__name__}: {use_exc}",
                flush=True,
            )

    tried = ", ".join(ref for ref, _ in errors)
    raise RuntimeError(f"Could not find a history artifact for run {run_id}. Tried: {tried}")


def load_cached_full_history(run_name, run_id, idx=None, total=None, parquet_path=None, csv_path=None):
    idx = idx or 1
    total = total or 1
    parquet_path = Path(parquet_path) if parquet_path else full_parquet_path(run_name, run_id)
    csv_path = Path(csv_path) if csv_path else full_csv_path(run_name, run_id)
    t0 = time.time()
    print(_progress("loading artifact cache", idx, total, run_name), flush=True)
    history = _read_history_table(parquet_path, csv_path=csv_path)
    source_path = history.attrs.get("source_path", str(parquet_path if parquet_path.exists() else csv_path))
    history = _ensure_run_columns(history, run_name, run_id)
    print(
        f"    loaded {len(history):,} rows, {len(history.columns):,} columns "
        f"from {source_path} in {time.time() - t0:.1f}s",
        flush=True,
    )
    return history


def download_run_full_history_from_artifact(row, idx=None, total=None):
    idx = idx or 1
    total = total or 1
    run_name = _row_get(row, "name")
    run_id = _row_get(row, "id")
    state = _row_get(row, "state")
    artifact_ref = artifact_path_for_run(run_id)
    cache_dir = run_cache_dir(run_name, run_id)
    parquet_path = full_parquet_path(run_name, run_id)
    csv_path = full_csv_path(run_name, run_id)
    meta_path = metadata_path(run_name, run_id)

    if parquet_path.exists() and not FORCE_REFRESH:
        return load_cached_full_history(run_name, run_id, idx=idx, total=total)

    cache_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print(_progress("downloading history artifact", idx, total, f"{run_name} ({artifact_ref})"), flush=True)
    artifact, resolved_artifact_ref = _get_history_artifact(run_id)
    download_dir = Path(artifact.download(root=str(cache_dir)))
    downloaded_parquet = _find_downloaded_history(download_dir)
    if downloaded_parquet.resolve() != parquet_path.resolve():
        shutil.copy2(downloaded_parquet, parquet_path)

    history = _read_history_table(parquet_path, csv_path=csv_path)
    history_with_ids = _ensure_run_columns(history, run_name, run_id)
    if WRITE_FULL_HISTORY_CSV:
        history_with_ids.to_csv(csv_path, index=False)

    metadata = {
        "name": run_name,
        "id": run_id,
        "state": state,
        "entity": ENTITY,
        "project": PROJECT,
        "artifact_requested": artifact_ref,
        "artifact_resolved": resolved_artifact_ref,
        "history_parquet": str(parquet_path),
        "history_csv": str(csv_path) if WRITE_FULL_HISTORY_CSV else None,
        "rows": int(len(history_with_ids)),
        "columns": int(len(history_with_ids.columns)),
        "downloaded_at_utc": pd.Timestamp.utcnow().isoformat(),
    }
    meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(
        f"    saved {len(history_with_ids):,} rows, {len(history_with_ids.columns):,} columns "
        f"in {time.time() - t0:.1f}s",
        flush=True,
    )
    return history_with_ids


def full_history_to_long(full_history, metrics):
    if full_history.empty:
        return pd.DataFrame(columns=["run_name", "run_id", "metric", "step", "value"])
    step_col = "_step" if "_step" in full_history.columns else "step"
    if step_col not in full_history.columns:
        raise ValueError("History has neither '_step' nor 'step' column.")
    available_metrics = [metric for metric in metrics if metric in full_history.columns]
    if not available_metrics:
        return pd.DataFrame(columns=["run_name", "run_id", "metric", "step", "value"])

    id_cols = [col for col in ["run_name", "run_id", step_col] if col in full_history.columns]
    long_history = full_history[id_cols + available_metrics].melt(
        id_vars=id_cols,
        value_vars=available_metrics,
        var_name="metric",
        value_name="value",
    )
    long_history = long_history.dropna(subset=["value"])
    if step_col != "step":
        long_history = long_history.rename(columns={step_col: "step"})
    return long_history[["run_name", "run_id", "metric", "step", "value"]]


def _runs_df_count():
    return len(runs_df) if "runs_df" in globals() else 0


def _check_download_inputs():
    runs_count = _runs_df_count()
    print(
        f"History artifact setup: local_only={USE_LOCAL_CACHE_ONLY}, runs_df={runs_count}",
        flush=True,
    )
    if runs_count == 0:
        raise RuntimeError(
            "runs_df is empty. Rerun the 'Fetch Run List' cell after changing "
            "RUN_NAME_REGEX/RUN_STATES, then rerun this cell."
        )


def download_or_load_histories():
    if CACHE_MODE != "full":
        raise ValueError('This notebook now uses artifact full-history caching. Set CACHE_MODE = "full".')

    _check_download_inputs()
    t0 = time.time()
    histories = []
    cache_index_rows = []
    skipped_rows = []
    total = len(runs_df)
    for idx, row in enumerate(runs_df.itertuples(index=False), start=1):
        run_name = _row_get(row, "name")
        run_id = _row_get(row, "id")
        try:
            if USE_LOCAL_CACHE_ONLY:
                full_history = load_cached_full_history(
                    run_name,
                    run_id,
                    idx=idx,
                    total=total,
                    parquet_path=_row_get(row, "history_parquet"),
                    csv_path=_row_get(row, "history_csv"),
                )
            else:
                full_history = download_run_full_history_from_artifact(row, idx=idx, total=total)
            histories.append(full_history_to_long(full_history, METRICS))
            resolved_ref = artifact_path_for_run(run_id)
            if metadata_path(run_name, run_id).exists():
                try:
                    resolved_ref = json.loads(metadata_path(run_name, run_id).read_text(encoding="utf-8")).get("artifact_resolved", resolved_ref)
                except json.JSONDecodeError:
                    pass
            cache_index_rows.append({
                "name": run_name,
                "id": run_id,
                "artifact": resolved_ref,
                "history_parquet": str(full_parquet_path(run_name, run_id)),
                "history_csv": str(full_csv_path(run_name, run_id)) if WRITE_FULL_HISTORY_CSV else None,
                "rows": len(full_history),
                "columns": len(full_history.columns),
            })
        except Exception as exc:
            print(f"    SKIPPED {run_name}: {type(exc).__name__}: {exc}", flush=True)
            skipped_rows.append({
                "name": run_name,
                "id": run_id,
                "state": _row_get(row, "state"),
                "error_type": type(exc).__name__,
                "error": str(exc),
            })
            if not SKIP_FAILED_DOWNLOADS:
                raise
    cache_index = pd.DataFrame(cache_index_rows)
    cache_index.to_csv(CACHE_INDEX_PATH, index=False)
    skipped_df = pd.DataFrame(skipped_rows)
    skipped_df.to_csv(FAILED_HISTORY_DOWNLOADS_PATH, index=False)
    print(f"Finished history artifact loading in {time.time() - t0:.1f}s", flush=True)
    print(f"Loaded histories for {len(histories)} / {total} selected runs", flush=True)
    if skipped_rows:
        print(f"Skipped {len(skipped_rows)} run(s); saved details: {FAILED_HISTORY_DOWNLOADS_PATH}", flush=True)
    print(f"Saved cache index: {CACHE_INDEX_PATH}", flush=True)
    return histories, skipped_df


histories, skipped_history_downloads_df = download_or_load_histories()
history_df = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame(columns=["run_name", "run_id", "metric", "step", "value"])
history_df["step"] = pd.to_numeric(history_df["step"], errors="coerce")
history_df["value"] = pd.to_numeric(history_df["value"], errors="coerce")
history_df = history_df.dropna(subset=["step", "value"])
history_df["step_millions"] = history_df["step"] / 1_000_000
print(history_df.shape)
history_df.head()


History artifact setup: local_only=True, runs_df=126
[ 1/126] loading artifact cache: MAPPO_baseline_nejjar
    loaded 625 rows, 30 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history/MAPPO_baseline_nejjar__MAPPO_bus14_T_0_0__I__1779543882_26350/history.parquet in 0.1s
[ 2/126] loading artifact cache: a0_known_good_det
    loaded 375 rows, 30 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history/a0_known_good_det__MAPPO_bus14_T_0_0__I__1779591426_12887/history.parquet in 0.0s
[ 3/126] loading artifact cache: a0_known_good_sto
    loaded 375 rows, 30 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history/a0_known_good_sto__MAPPO_bus14_T_0_0__I__1779592809_45825/history.parquet in 0.0s
[ 4/126] loading artifact cache: a0_stoch_20env_mb20_ep15_kl004
    loaded 375 rows, 32 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topolog

,run_name,run_id,metric,step,value,step_millions
0,MAPPO_baseline_nejjar,MAPPO_bus14_T_0_0__I__1779543882_26350,charts/episodic_survival,80000.0,0.000322,0.08
1,MAPPO_baseline_nejjar,MAPPO_bus14_T_0_0__I__1779543882_26350,charts/episodic_survival,160000.0,0.000533,0.16
2,MAPPO_baseline_nejjar,MAPPO_bus14_T_0_0__I__1779543882_26350,charts/episodic_survival,240000.0,0.000620,0.24
3,MAPPO_baseline_nejjar,MAPPO_bus14_T_0_0__I__1779543882_26350,charts/episodic_survival,320000.0,0.000868,0.32
4,MAPPO_baseline_nejjar,MAPPO_bus14_T_0_0__I__1779543882_26350,charts/episodic_survival,400000.0,0.002517,0.40


## Simple Plotting

Run the W&B data cells above first. Then use `plot_runs(...)` for one chart, or `plot_run_groups(...)` for subplots.


In [6]:
SURVIVAL_METRIC_CANDIDATES = {
    "test": [
        "test/charts/episodic_survival",
        "test/episodic_survival",
        "charts/episodic_survival",
    ],
    "train_eval": [
        "train_eval/charts/episodic_survival",
        "train_eval/episodic_survival",
    ],
    "validation": ["validation/episodic_survival"],
    "legacy": ["charts/episodic_survival"],
}

DEFAULT_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
]


def available_run_names(pattern=None, history=None):
    """Return cached run names, optionally filtered by a regex pattern."""
    history = _get_history(history)
    names = sorted(history["run_name"].dropna().unique())
    if pattern is None:
        return names
    regex = re.compile(pattern)
    return [name for name in names if regex.search(name)]


def make_run(name, color=None, label=None, dash=None):
    """Small helper for explicit run specs."""
    return {"name": name, "color": color, "label": label, "dash": dash}


def save_plot(fig, name):
    """Save a Plotly figure as HTML under FIG_DIR, unless an absolute path is passed."""
    if not name:
        return None
    path = Path(name)
    if path.suffix.lower() != ".html":
        path = FIG_DIR / f"{safe_name(name)}.html"
    elif not path.is_absolute():
        path = FIG_DIR / path
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(path, include_plotlyjs="cdn")
    print(f"Saved plot: {path}")
    return path


def plot_runs(
    runs,
    *,
    metric=None,
    split="test",
    colors=None,
    labels=None,
    dashes=None,
    smooth=50,
    show_raw=True,
    multiply=100.0,
    title=None,
    yaxis_title="Survival (%)",
    xaxis_title="Steps (M)",
    y_range=None,
    width=1300,
    height=550,
    save_name=None,
    show=False,
    history=None,
):
    """
    Plot one chart with all requested runs.

    `runs` can be:
    - {"run name": "#color", ...}
    - [("run name", "#color", "label", "dash"), ...]
    - [{"name": "run name", "color": "#color", "label": "label", "dash": "dash"}, ...]
    """
    history = _get_history(history)
    specs = _normalize_runs(runs, colors=colors, labels=labels, dashes=dashes)
    metric_candidates = _metric_candidates(metric=metric, split=split)

    fig = go.Figure()
    added = _add_run_traces(
        fig,
        specs,
        metric_candidates,
        history=history,
        smooth=smooth,
        show_raw=show_raw,
        multiply=multiply,
        legend_seen=set(),
    )
    if added == 0:
        fig.add_annotation(
            text="No data found for the requested runs and metric.",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False,
        )

    fig.update_layout(
        title=title,
        template="plotly_white",
        width=width,
        height=height,
        hovermode="x unified",
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0},
        margin={"l": 70, "r": 30, "t": 80, "b": 60},
    )
    fig.update_xaxes(title_text=xaxis_title)
    fig.update_yaxes(title_text=yaxis_title, range=y_range)
    save_plot(fig, save_name)
    if show:
        fig.show()
        return None
    return fig


def plot_run_groups(
    groups,
    *,
    metric=None,
    split="test",
    colors=None,
    labels=None,
    dashes=None,
    smooth=50,
    show_raw=True,
    multiply=100.0,
    title=None,
    yaxis_title="Survival (%)",
    xaxis_title="Steps (M)",
    y_range=None,
    ncols=2,
    subplot_height=420,
    width=1500,
    save_name=None,
    show=False,
    history=None,
):
    """
    Plot several run groups as subplots.

    `groups` can be a dict like {"subplot title": runs, ...}, where each `runs`
    value accepts the same formats as `plot_runs`.
    """
    history = _get_history(history)
    group_items = _normalize_groups(groups)
    if not group_items:
        raise ValueError("Pass at least one subplot group.")

    ncols = max(1, int(ncols))
    nrows = int(np.ceil(len(group_items) / ncols))
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=[title for title, _ in group_items],
        shared_xaxes=False,
        shared_yaxes=False,
    )
    metric_candidates = _metric_candidates(metric=metric, split=split)
    added = 0
    legend_layouts = {}

    for idx, (_, group_runs) in enumerate(group_items, start=1):
        row = int(np.ceil(idx / ncols))
        col = ((idx - 1) % ncols) + 1
        specs = _normalize_runs(group_runs, colors=colors, labels=labels, dashes=dashes)
        legend_id = _legend_id(idx)
        legend_layouts[legend_id] = _subplot_legend_layout(fig, row, col, ncols)
        added += _add_run_traces(
            fig,
            specs,
            metric_candidates,
            history=history,
            smooth=smooth,
            show_raw=show_raw,
            multiply=multiply,
            row=row,
            col=col,
            legend_seen=set(),
            legend_id=legend_id,
            legend_group_prefix=f"subplot{idx}:",
        )
        fig.update_xaxes(title_text=xaxis_title, row=row, col=col)
        fig.update_yaxes(title_text=yaxis_title, range=y_range, row=row, col=col)

    if added == 0:
        fig.add_annotation(
            text="No data found for the requested runs and metric.",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False,
        )

    layout = {
        "title": title,
        "template": "plotly_white",
        "width": width,
        "height": max(520, subplot_height * nrows),
        "hovermode": "x unified",
        "showlegend": True,
        "margin": {"l": 70, "r": 30, "t": 90, "b": 60},
    }
    layout.update(legend_layouts)
    fig.update_layout(**layout)
    save_plot(fig, save_name)
    if show:
        fig.show()
        return None
    return fig


def plot_run_means(
    mean_runs,
    *,
    metric=None,
    split="test",
    colors=None,
    dashes=None,
    smooth=50,
    show_members=False,
    show_std=True,
    min_members=1,
    multiply=100.0,
    title=None,
    yaxis_title="Survival (%)",
    xaxis_title="Steps (M)",
    y_range=None,
    width=1300,
    height=550,
    save_name=None,
    show=False,
    history=None,
):
    """
    Plot averaged curves. Each entry in `mean_runs` is one curve.

    Example spec:
    {
        "entropy_decay_sto": {"runs": ["run_s0_stoch", "run_s1_stoch"], "color": "#1f77b4"},
        "entropy_decay_det": {"runs": ["run_s0_det", "run_s1_det"], "color": "#1f77b4", "dash": "dash"},
        "no_entropy_decay_sto": {"runs": ["run_s0_stoch", "run_s1_stoch"], "color": "#ff7f0e"},
    }
    """
    history = _get_history(history)
    specs = _normalize_mean_specs(mean_runs, colors=colors, dashes=dashes)
    metric_candidates = _metric_candidates(metric=metric, split=split)

    fig = go.Figure()
    added = _add_mean_traces(
        fig,
        specs,
        metric_candidates,
        history=history,
        smooth=smooth,
        show_members=show_members,
        show_std=show_std,
        min_members=min_members,
        multiply=multiply,
        legend_seen=set(),
    )
    if added == 0:
        fig.add_annotation(
            text="No data found for the requested mean curves.",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False,
        )

    fig.update_layout(
        title=title,
        template="plotly_white",
        width=width,
        height=height,
        hovermode="x unified",
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0},
        margin={"l": 70, "r": 30, "t": 80, "b": 60},
    )
    fig.update_xaxes(title_text=xaxis_title)
    fig.update_yaxes(title_text=yaxis_title, range=y_range)
    save_plot(fig, save_name)
    if show:
        fig.show()
        return None
    return fig


def plot_run_mean_groups(
    groups,
    *,
    metric=None,
    split="test",
    colors=None,
    dashes=None,
    smooth=50,
    show_members=False,
    show_std=True,
    min_members=1,
    multiply=100.0,
    title=None,
    yaxis_title="Survival (%)",
    xaxis_title="Steps (M)",
    y_range=None,
    ncols=2,
    subplot_height=420,
    width=1500,
    save_name=None,
    show=False,
    history=None,
):
    """
    Plot mean curves in subplots.

    `groups` is a dict like {"subplot title": mean_runs, ...}. Each `mean_runs`
    value accepts the same format as `plot_run_means`.
    """
    history = _get_history(history)
    group_items = _normalize_groups(groups)
    if not group_items:
        raise ValueError("Pass at least one subplot group.")

    ncols = max(1, int(ncols))
    nrows = int(np.ceil(len(group_items) / ncols))
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=[title for title, _ in group_items],
        shared_xaxes=False,
        shared_yaxes=False,
    )
    metric_candidates = _metric_candidates(metric=metric, split=split)
    added = 0
    legend_layouts = {}

    for idx, (_, mean_runs) in enumerate(group_items, start=1):
        row = int(np.ceil(idx / ncols))
        col = ((idx - 1) % ncols) + 1
        specs = _normalize_mean_specs(mean_runs, colors=colors, dashes=dashes)
        legend_id = _legend_id(idx)
        legend_layouts[legend_id] = _subplot_legend_layout(fig, row, col, ncols)
        added += _add_mean_traces(
            fig,
            specs,
            metric_candidates,
            history=history,
            smooth=smooth,
            show_members=show_members,
            show_std=show_std,
            min_members=min_members,
            multiply=multiply,
            row=row,
            col=col,
            legend_seen=set(),
            legend_id=legend_id,
            legend_group_prefix=f"mean_subplot{idx}:",
        )
        fig.update_xaxes(title_text=xaxis_title, row=row, col=col)
        fig.update_yaxes(title_text=yaxis_title, range=y_range, row=row, col=col)

    if added == 0:
        fig.add_annotation(
            text="No data found for the requested mean curves.",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False,
        )

    layout = {
        "title": title,
        "template": "plotly_white",
        "width": width,
        "height": max(520, subplot_height * nrows),
        "hovermode": "x unified",
        "showlegend": True,
        "margin": {"l": 70, "r": 30, "t": 90, "b": 60},
    }
    layout.update(legend_layouts)
    fig.update_layout(**layout)
    save_plot(fig, save_name)
    if show:
        fig.show()
        return None
    return fig


def _get_history(history=None):
    if history is not None:
        return history
    if "history_df" not in globals():
        raise NameError("Run the W&B history loading cell first so `history_df` exists.")
    return history_df


def _metric_candidates(metric=None, split="test"):
    if metric is None:
        return SURVIVAL_METRIC_CANDIDATES.get(split, SURVIVAL_METRIC_CANDIDATES["test"])
    if isinstance(metric, str):
        return [metric]
    return list(metric)


def _normalize_groups(groups):
    if isinstance(groups, dict):
        return list(groups.items())
    normalized = []
    for item in groups:
        if isinstance(item, dict):
            normalized.append((item["title"], item["runs"]))
        else:
            title, runs = item
            normalized.append((title, runs))
    return normalized


def _legend_id(index):
    return "legend" if index == 1 else f"legend{index}"


def _subplot_legend_layout(fig, row, col, ncols):
    x_domain, y_domain = _subplot_domains(fig, row, col, ncols)
    return {
        "x": x_domain[0] + 0.01,
        "y": y_domain[1] - 0.02,
        "xanchor": "left",
        "yanchor": "top",
        "orientation": "v",
        "bgcolor": "rgba(255,255,255,0.62)",
        "bordercolor": "rgba(0,0,0,0.10)",
        "borderwidth": 1,
        "font": {"size": 9},
        "itemsizing": "constant",
    }


def _subplot_domains(fig, row, col, ncols):
    axis_index = (row - 1) * ncols + col
    suffix = "" if axis_index == 1 else str(axis_index)
    x_axis = getattr(fig.layout, f"xaxis{suffix}")
    y_axis = getattr(fig.layout, f"yaxis{suffix}")
    return x_axis.domain, y_axis.domain


def mean_curve(
    runs,
    *,
    color=None,
    dash=None,
    label=None,
    width=3,
    opacity=1.0,
    show_std=None,
    std_alpha=0.14,
    show_members=None,
    member_alpha=0.18,
):
    """Create one averaged-curve spec for `plot_run_means`."""
    spec = {
        "runs": runs,
        "color": color,
        "dash": dash,
        "width": width,
        "opacity": opacity,
        "show_std": show_std,
        "std_alpha": std_alpha,
        "show_members": show_members,
        "member_alpha": member_alpha,
    }
    if label is not None:
        spec["label"] = label
    return spec


def mean_curves_from_prefixes(
    entropy_prefix,
    no_entropy_prefix,
    *,
    seeds=(0, 1, 2),
    entropy_label="entropy decay",
    no_entropy_label="no entropy decay",
    entropy_color="#1f77b4",
    no_entropy_color="#ff7f0e",
    stoch_suffix="stoch",
    det_suffix="det",
    stoch_dash="solid",
    det_dash="dash",
    **style,
):
    """Build four mean curves from exact W&B run-name prefixes."""
    return {
        f"{entropy_label} sto": mean_curve(
            _seeded_run_names(entropy_prefix, seeds, stoch_suffix),
            color=entropy_color,
            dash=stoch_dash,
            **style,
        ),
        f"{entropy_label} det": mean_curve(
            _seeded_run_names(entropy_prefix, seeds, det_suffix),
            color=entropy_color,
            dash=det_dash,
            **style,
        ),
        f"{no_entropy_label} sto": mean_curve(
            _seeded_run_names(no_entropy_prefix, seeds, stoch_suffix),
            color=no_entropy_color,
            dash=stoch_dash,
            **style,
        ),
        f"{no_entropy_label} det": mean_curve(
            _seeded_run_names(no_entropy_prefix, seeds, det_suffix),
            color=no_entropy_color,
            dash=det_dash,
            **style,
        ),
    }


def _seeded_run_names(prefix, seeds, suffix):
    return [f"{prefix}_s{seed}_{suffix}" for seed in seeds]


def sto_det_mean_specs(
    runs,
    *,
    color="#1f77b4",
    det_color=None,
    stoch_label="sto",
    det_label="det",
    stoch_dash="solid",
    det_dash="dash",
    **style,
):
    """Optional helper: split a mixed run list into stochastic and deterministic mean specs."""
    run_names = _as_run_name_list(runs)
    stoch_runs = [name for name in run_names if _is_stoch_run(name)]
    det_runs = [name for name in run_names if _is_det_run(name)]
    specs = {}
    if stoch_runs:
        specs[stoch_label] = mean_curve(stoch_runs, color=color, dash=stoch_dash, **style)
    if det_runs:
        specs[det_label] = mean_curve(det_runs, color=det_color or color, dash=det_dash, **style)
    return specs


def _normalize_mean_specs(mean_runs, colors=None, dashes=None):
    if isinstance(mean_runs, dict):
        raw_specs = [_coerce_mean_spec(label, value) for label, value in mean_runs.items()]
    else:
        raw_specs = [_coerce_mean_spec_from_item(item) for item in mean_runs]

    specs = []
    for idx, spec in enumerate(raw_specs):
        label = spec["label"]
        spec["runs"] = _as_run_name_list(spec["runs"])
        spec["color"] = spec.get("color") or _lookup(colors, label, idx) or DEFAULT_COLORS[idx % len(DEFAULT_COLORS)]
        spec["dash"] = spec.get("dash") or _lookup(dashes, label, idx) or _auto_mean_dash(label)
        spec["width"] = spec.get("width", spec.get("line_width", 3))
        spec["opacity"] = spec.get("opacity", 1.0)
        spec["std_alpha"] = spec.get("std_alpha", 0.14)
        spec["member_alpha"] = spec.get("member_alpha", 0.18)
        specs.append(spec)
    return specs


def _coerce_mean_spec(label, value):
    if isinstance(value, dict):
        spec = dict(value)
        spec.setdefault("label", label)
        if "runs" not in spec:
            style_keys = {
                "label", "name", "color", "dash", "width", "line_width", "opacity",
                "show_std", "std_alpha", "show_members", "member_alpha",
            }
            if any(key in spec for key in style_keys):
                raise ValueError(f"Mean curve {label!r} has style keys but no `runs` list.")
            spec["runs"] = list(value.keys())
    else:
        spec = {"label": label, "runs": value}
    if "name" in spec and "label" not in spec:
        spec["label"] = spec["name"]
    return spec


def _coerce_mean_spec_from_item(item):
    if isinstance(item, dict):
        spec = dict(item)
        if "name" in spec and "label" not in spec:
            spec["label"] = spec["name"]
        if "label" not in spec or "runs" not in spec:
            raise ValueError("Mean specs need `label` and `runs` keys.")
        return spec
    if isinstance(item, (tuple, list)) and len(item) >= 2:
        spec = {"label": item[0], "runs": item[1]}
        if len(item) > 2:
            spec["color"] = item[2]
        if len(item) > 3:
            spec["dash"] = item[3]
        if len(item) > 4:
            spec["width"] = item[4]
        return spec
    raise ValueError(f"Could not understand mean spec: {item!r}")


def _as_run_name_list(runs):
    if isinstance(runs, str):
        return [runs]
    if isinstance(runs, dict):
        return list(runs.keys())
    return list(runs)


def _auto_mean_dash(label):
    return "dash" if "det" in str(label).lower() else "solid"


def _is_det_run(name):
    lower = str(name).lower()
    return lower.endswith("_det") or "_det_" in lower


def _is_stoch_run(name):
    lower = str(name).lower()
    return lower.endswith("_stoch") or "_stoch_" in lower or lower.endswith("_sto") or "_sto_" in lower


def _mean_metric_frame(history, spec, metric_candidates, min_members=1):
    frames = []
    metrics_used = []
    for run_name in spec["runs"]:
        data, metric = _run_metric_frame(history, run_name, metric_candidates)
        if data.empty:
            continue
        member = data[["step", "step_millions", "value"]].copy()
        member["run_name"] = run_name
        frames.append(member)
        metrics_used.append(metric)

    if not frames:
        print(f"Skipping {spec['label']}: no usable runs found")
        return pd.DataFrame(), pd.DataFrame(), []

    if len(frames) < len(spec["runs"]):
        print(f"{spec['label']}: using {len(frames)} / {len(spec['runs'])} runs")

    members = pd.concat(frames, ignore_index=True)
    stats = members.groupby("step", as_index=False).agg(
        mean=("value", "mean"),
        std=("value", "std"),
        n=("value", "count"),
    )
    stats = stats[stats["n"] >= int(min_members)].copy()
    if stats.empty:
        print(f"Skipping {spec['label']}: no step has at least {min_members} member(s)")
        return pd.DataFrame(), members, metrics_used
    stats["std"] = stats["std"].fillna(0.0)
    stats["step_millions"] = stats["step"] / 1_000_000
    return stats.sort_values("step"), members.sort_values(["run_name", "step"]), metrics_used


def _add_mean_traces(
    fig,
    specs,
    metric_candidates,
    *,
    history,
    smooth,
    show_members,
    show_std,
    min_members,
    multiply,
    row=None,
    col=None,
    legend_seen=None,
    legend_id="legend",
    legend_group_prefix="",
):
    legend_seen = legend_seen if legend_seen is not None else set()
    added = 0
    add_kwargs = {} if row is None else {"row": row, "col": col}

    for spec in specs:
        label = spec["label"]
        color = spec["color"]
        dash = spec["dash"]
        width = spec["width"]
        opacity = spec["opacity"]
        member_alpha = spec["member_alpha"]
        std_alpha = spec["std_alpha"]
        use_members = show_members if spec.get("show_members") is None else spec["show_members"]
        use_std = show_std if spec.get("show_std") is None else spec["show_std"]
        stats, members, metrics_used = _mean_metric_frame(history, spec, metric_candidates, min_members=min_members)
        if stats.empty:
            continue

        legend_group = f"{legend_group_prefix}{label}"
        if use_members:
            for run_name, member in members.groupby("run_name"):
                fig.add_trace(
                    go.Scatter(
                        x=member["step_millions"],
                        y=_smooth_values(_scale_values(member["value"], multiply), smooth),
                        mode="lines",
                        name=f"{label} member",
                        legendgroup=legend_group,
                        legend=legend_id,
                        showlegend=False,
                        opacity=member_alpha,
                        line={"color": color, "width": spec.get("member_width", 1), "dash": dash},
                        hovertemplate=(
                            f"<b>{run_name}</b><br>"
                            "step: %{x:.2f}M<br>"
                            "value: %{y:.3f}<extra></extra>"
                        ),
                    ),
                    **add_kwargs,
                )
                added += 1

        x = stats["step_millions"]
        mean_y = _smooth_values(_scale_values(stats["mean"], multiply), smooth)
        std_y = _smooth_values(_scale_values(stats["std"], multiply), smooth)

        if use_std and stats["n"].max() > 1:
            upper = mean_y + std_y
            lower = mean_y - std_y
            fig.add_trace(
                go.Scatter(
                    x=list(x) + list(x[::-1]),
                    y=list(upper) + list(lower[::-1]),
                    mode="lines",
                    name=f"{label} std",
                    legendgroup=legend_group,
                    legend=legend_id,
                    showlegend=False,
                    line={"color": "rgba(0,0,0,0)", "width": 0},
                    fill="toself",
                    fillcolor=_color_with_alpha(color, std_alpha),
                    hoverinfo="skip",
                ),
                **add_kwargs,
            )
            added += 1

        legend_key = (label, color, dash)
        show_mean_legend = legend_key not in legend_seen
        legend_seen.add(legend_key)
        metric_label = ", ".join(sorted(set(metrics_used)))
        fig.add_trace(
            go.Scatter(
                x=x,
                y=mean_y,
                mode="lines",
                name=label,
                legendgroup=legend_group,
                legend=legend_id,
                showlegend=show_mean_legend,
                customdata=stats["n"],
                opacity=opacity,
                line={"color": color, "width": width, "dash": dash},
                hovertemplate=(
                    f"<b>{label}</b><br>"
                    f"metric: {metric_label}<br>"
                    "runs at step: %{customdata}<br>"
                    "step: %{x:.2f}M<br>"
                    "mean: %{y:.3f}<extra></extra>"
                ),
            ),
            **add_kwargs,
        )
        added += 1
    return added


def _color_with_alpha(color, alpha):
    color = str(color)
    match = re.fullmatch(r"#?([0-9A-Fa-f]{6})", color)
    if not match:
        return f"rgba(127,127,127,{alpha})"
    value = match.group(1)
    red = int(value[0:2], 16)
    green = int(value[2:4], 16)
    blue = int(value[4:6], 16)
    return f"rgba({red},{green},{blue},{alpha})"


def _normalize_runs(runs, colors=None, labels=None, dashes=None):
    if isinstance(runs, dict):
        run_items = [{"name": name, "color": color} for name, color in runs.items()]
    else:
        run_items = list(runs)

    specs = []
    for idx, item in enumerate(run_items):
        spec = _coerce_run_spec(item)
        name = spec["name"]
        spec["color"] = spec.get("color") or _lookup(colors, name, idx) or DEFAULT_COLORS[idx % len(DEFAULT_COLORS)]
        spec["label"] = spec.get("label") or _lookup(labels, name, idx) or name
        spec["dash"] = spec.get("dash") or _lookup(dashes, name, idx) or _auto_dash(name)
        specs.append(spec)
    return specs


def _coerce_run_spec(item):
    if isinstance(item, str):
        return {"name": item}
    if isinstance(item, dict):
        if "name" in item:
            return dict(item)
        if len(item) == 1:
            name, color = next(iter(item.items()))
            return {"name": name, "color": color}
        raise ValueError("Run dictionaries need a `name` key, or exactly one {name: color} item.")
    if isinstance(item, (tuple, list)) and item:
        spec = {"name": item[0]}
        if len(item) > 1:
            spec["color"] = item[1]
        if len(item) > 2:
            spec["label"] = item[2]
        if len(item) > 3:
            spec["dash"] = item[3]
        return spec
    raise ValueError(f"Could not understand run spec: {item!r}")


def _lookup(values, name, idx):
    if values is None:
        return None
    if isinstance(values, dict):
        return values.get(name)
    values = list(values)
    if not values:
        return None
    return values[idx % len(values)]


def _auto_dash(name):
    lower = str(name).lower()
    return "dash" if lower.endswith("_det") or "_det_" in lower else "solid"


def _run_metric_frame(history, run_name, metric_candidates):
    run_history = history[history["run_name"] == run_name]
    if run_history.empty:
        print(f"Skipping {run_name}: run not found in history_df")
        return pd.DataFrame(), None

    for metric in metric_candidates:
        metric_history = run_history[run_history["metric"] == metric].copy()
        if metric_history.empty:
            continue
        metric_history["step"] = pd.to_numeric(metric_history["step"], errors="coerce")
        metric_history["value"] = pd.to_numeric(metric_history["value"], errors="coerce")
        metric_history = metric_history.dropna(subset=["step", "value"])
        if metric_history.empty:
            continue
        metric_history = metric_history.groupby("step", as_index=False)["value"].mean()
        metric_history["step_millions"] = metric_history["step"] / 1_000_000
        return metric_history.sort_values("step"), metric

    print(f"Skipping {run_name}: none of these metrics were found: {metric_candidates}")
    return pd.DataFrame(), None


def _smooth_values(values, smooth):
    if smooth is None or int(smooth) <= 1:
        return values
    return values.rolling(int(smooth), min_periods=1).mean()


def _scale_values(values, multiply):
    values = pd.to_numeric(values, errors="coerce")
    if multiply is not None:
        values = values * float(multiply)
    return values


def _add_run_traces(
    fig,
    specs,
    metric_candidates,
    *,
    history,
    smooth,
    show_raw,
    multiply,
    row=None,
    col=None,
    legend_seen=None,
    legend_id="legend",
    legend_group_prefix="",
):
    legend_seen = legend_seen if legend_seen is not None else set()
    added = 0
    add_kwargs = {} if row is None else {"row": row, "col": col}

    for spec in specs:
        run_name = spec["name"]
        label = spec["label"]
        color = spec["color"]
        dash = spec["dash"]
        data, metric = _run_metric_frame(history, run_name, metric_candidates)
        if data.empty:
            continue

        x = data["step_millions"]
        y = _scale_values(data["value"], multiply)
        hover = (
            "<b>%{fullData.name}</b><br>"
            "metric: " + metric + "<br>"
            "step: %{x:.2f}M<br>"
            "value: %{y:.3f}<extra></extra>"
        )
        legend_group = f"{legend_group_prefix}{label}"

        if show_raw:
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=y,
                    mode="lines",
                    name=f"{label} raw",
                    legendgroup=legend_group,
                    legend=legend_id,
                    showlegend=False,
                    opacity=0.22,
                    line={"color": color, "width": 1, "dash": dash},
                    hovertemplate=hover,
                ),
                **add_kwargs,
            )
            added += 1

        legend_key = (label, color, dash)
        show_smooth_legend = legend_key not in legend_seen
        legend_seen.add(legend_key)
        fig.add_trace(
            go.Scatter(
                x=x,
                y=_smooth_values(y, smooth),
                mode="lines",
                name=label,
                legendgroup=legend_group,
                legend=legend_id,
                showlegend=show_smooth_legend,
                line={"color": color, "width": 2.8, "dash": dash},
                hovertemplate=hover,
            ),
            **add_kwargs,
        )
        added += 1
    return added


## Examples

Edit the run names/colors and uncomment the call you want. Deterministic runs get a dashed line automatically when the run name ends with `_det`.


In [ ]:
# See which runs are currently loaded.
# available_run_names("entropy_decay")

# One plot. Dict form means {run_name: color}.
# plot_runs(
#     {
#         "noval20_mlp_a1_entropy_decay_s0_stoch": "#1f77b4",
#         "noval20_mlp_a1_entropy_decay_s0_det": "#1f77b4",
#         "noval20_mlp_a1_entropy_decay_s1_stoch": "#ff7f0e",
#         "noval20_mlp_a1_entropy_decay_s1_det": "#ff7f0e",
#     },
#     split="test",
#     smooth=50,
#     title="A1 entropy decay",
#     save_name="a1_entropy_decay_test",
# )

# Subplots. Each value accepts the same run formats as plot_runs.
# groups = {
#     "A1": {
#         "noval20_mlp_a1_entropy_decay_s0_stoch": "#1f77b4",
#         "noval20_mlp_a1_entropy_decay_s0_det": "#1f77b4",
#     },
#     "A3": {
#         "noval20_mlp_a3_entropy_decay_s0_stoch": "#ff7f0e",
#         "noval20_mlp_a3_entropy_decay_s0_det": "#ff7f0e",
#     },
# }
# plot_run_groups(groups, split="test", smooth=50, title="Entropy decay seed sweep", save_name="entropy_decay_subplots")

# For another metric, pass the exact metric name and usually multiply=1.
# plot_runs(groups["A1"], metric="train/entropy_coef", multiply=1, yaxis_title="Entropy coef")

In [8]:
groups = {
    "A1": {
        "noval20_mlp_a1_entropy_decay_s0_stoch": "#1f77b4",
        "noval20_mlp_a1_entropy_decay_s0_det": "#1f77b4",
        "noval20_mlp_a1_entropy_decay_s1_stoch": "#ff7f0e",
        "noval20_mlp_a1_entropy_decay_s1_det": "#ff7f0e",
        "noval20_mlp_a1_entropy_decay_s2_stoch": "#2ca02c",
        "noval20_mlp_a1_entropy_decay_s2_det": "#2ca02c",
    },
    "A3": {
        "noval20_mlp_a3_entropy_decay_s0_stoch": "#1f77b4",
        "noval20_mlp_a3_entropy_decay_s0_det": "#1f77b4",
        "noval20_mlp_a3_entropy_decay_s1_stoch": "#ff7f0e",
        "noval20_mlp_a3_entropy_decay_s1_det": "#ff7f0e",
        "noval20_mlp_a3_entropy_decay_s2_stoch": "#2ca02c",
        "noval20_mlp_a3_entropy_decay_s2_det": "#2ca02c",
    },
    "A4": {
        "noval20_mlp_a4_entropy_decay_s0_stoch": "#1f77b4",
        "noval20_mlp_a4_entropy_decay_s0_det": "#1f77b4",
        "noval20_mlp_a4_entropy_decay_s1_stoch": "#ff7f0e",
        "noval20_mlp_a4_entropy_decay_s1_det": "#ff7f0e",
        "noval20_mlp_a4_entropy_decay_s2_stoch": "#2ca02c",
        "noval20_mlp_a4_entropy_decay_s2_det": "#2ca02c",
    },
    "A3 logit": {
        "noval20_mlp_a3_logit_decay_s0_stoch": "#1f77b4",
        "noval20_mlp_a3_logit_decay_s0_det": "#1f77b4",
        "noval20_mlp_a3_logit_decay_s1_stoch": "#ff7f0e",
        "noval20_mlp_a3_logit_decay_s1_det": "#ff7f0e",
        "noval20_mlp_a3_logit_decay_s2_stoch": "#2ca02c",
        "noval20_mlp_a3_logit_decay_s2_det": "#2ca02c",
    },
    "p999": {
        "noval20_mlp_p999_decay_s0_stoch": "#1f77b4",
        "noval20_mlp_p999_decay_s0_det": "#1f77b4",
        "noval20_mlp_p999_decay_s1_stoch": "#ff7f0e",
        "noval20_mlp_p999_decay_s1_det": "#ff7f0e",
        "noval20_mlp_p999_decay_s2_stoch": "#2ca02c",
        "noval20_mlp_p999_decay_s2_det": "#2ca02c",
    }
}
plot_run_groups(groups, split="test", smooth=20, title="Entropy Decay", save_name="entropy_decay_subplots")

Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/entropy_decay_subplots.html


In [9]:
groups = {
    "A1": {
        "noval20_mlp_a1_no_entropy_decay_s0_stoch": "#1f77b4",
        "noval20_mlp_a1_no_entropy_decay_s0_det": "#1f77b4",
        "noval20_mlp_a1_no_entropy_decay_s1_stoch": "#ff7f0e",
        "noval20_mlp_a1_no_entropy_decay_s1_det": "#ff7f0e",
        "noval20_mlp_a1_no_entropy_decay_s2_stoch": "#2ca02c",
        "noval20_mlp_a1_no_entropy_decay_s2_det": "#2ca02c",
    },
    "A3": {
        "noval20_mlp_a3_no_entropy_decay_s0_stoch": "#1f77b4",
        "noval20_mlp_a3_no_entropy_decay_s0_det": "#1f77b4",
        "noval20_mlp_a3_no_entropy_decay_s1_stoch": "#ff7f0e",
        "noval20_mlp_a3_no_entropy_decay_s1_det": "#ff7f0e",
        "noval20_mlp_a3_no_entropy_decay_s2_stoch": "#2ca02c",
        "noval20_mlp_a3_no_entropy_decay_s2_det": "#2ca02c",
    },
    "A4": {
        "noval20_mlp_a4_no_entropy_decay_s0_stoch": "#1f77b4",
        "noval20_mlp_a4_no_entropy_decay_s0_det": "#1f77b4",
        "noval20_mlp_a4_no_entropy_decay_s1_stoch": "#ff7f0e",
        "noval20_mlp_a4_no_entropy_decay_s1_det": "#ff7f0e",
        "noval20_mlp_a4_no_entropy_decay_s2_stoch": "#2ca02c",
        "noval20_mlp_a4_no_entropy_decay_s2_det": "#2ca02c",
    },
    "A3 logit": {
        "noval20_mlp_a3_logit_no_entropy_decay_s0_stoch": "#1f77b4",
        "noval20_mlp_a3_logit_no_entropy_decay_s0_det": "#1f77b4",
        "noval20_mlp_a3_logit_no_entropy_decay_s1_stoch": "#ff7f0e",
        "noval20_mlp_a3_logit_no_entropy_decay_s1_det": "#ff7f0e",
        "noval20_mlp_a3_logit_no_entropy_decay_s2_stoch": "#2ca02c",
        "noval20_mlp_a3_logit_no_entropy_decay_s2_det": "#2ca02c",
    }
}
plot_run_groups(groups, split="test", smooth=20, title="No Entropy Decay", save_name="entropy_decay_subplots")

Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/entropy_decay_subplots.html


In [10]:
# Mean comparison for one setup.
mean_runs = {
    "sto": {
        "runs": [
            "noval20_mlp_a1_entropy_decay_s0_stoch",
            "noval20_mlp_a1_entropy_decay_s1_stoch",
            "noval20_mlp_a1_entropy_decay_s2_stoch",
        ],
        "color": "#1f77b4",
    },
    "det": {
        "runs": [
            "noval20_mlp_a1_entropy_decay_s0_det",
            "noval20_mlp_a1_entropy_decay_s1_det",
            "noval20_mlp_a1_entropy_decay_s2_det",
        ],
        "color": "#1f77b4",
        "dash": "dash",
    },
}
plot_run_means(mean_runs, split="test", smooth=50, title="A1 mean sto vs det")

In [13]:
# Generic averaged curves: every key is one mean curve.
# This can compare entropy/no-entropy and stochastic/deterministic in the same plot.
mean_runs = {
    "entropy decay sto": mean_curve(
        [
            "noval20_mlp_a1_entropy_decay_s0_stoch",
            "noval20_mlp_a1_entropy_decay_s1_stoch",
            "noval20_mlp_a1_entropy_decay_s2_stoch",
        ],
        color="#1f77b4",
        dash="solid",
    ),
    "entropy decay det": mean_curve(
        [
            "noval20_mlp_a1_entropy_decay_s0_det",
            "noval20_mlp_a1_entropy_decay_s1_det",
            "noval20_mlp_a1_entropy_decay_s2_det",
        ],
        color="#1f77b4",
        dash="dash",
    ),
    "no entropy decay sto": mean_curve(
        [
            "noval20_mlp_a1_no_entropy_decay_s0_stoch",
            "noval20_mlp_a1_no_entropy_decay_s1_stoch",
            "noval20_mlp_a1_no_entropy_decay_s2_stoch",
        ],
        color="#ff7f0e",
        dash="solid",
    ),
    "no entropy decay det": mean_curve(
        [
            "noval20_mlp_a1_no_entropy_decay_s0_det",
            "noval20_mlp_a1_no_entropy_decay_s1_det",
            "noval20_mlp_a1_no_entropy_decay_s2_det",
        ],
        color="#ff7f0e",
        dash="dash",
    ),
}
plot_run_means(mean_runs, split="test", smooth=50, title="A1 mean comparison")

In [19]:
# Generic averaged curves in subplots.
# Pass exact prefixes when a setup does not follow the same naming pattern.
mean_groups = {
    "A1": mean_curves_from_prefixes(
        "noval20_mlp_a1_entropy_decay",
        "noval20_mlp_a1_no_entropy_decay",
    ),
    "A3": mean_curves_from_prefixes(
        "noval20_mlp_a3_entropy_decay",
        "noval20_mlp_a3_no_entropy_decay",
    ),
    "A3 logit": mean_curves_from_prefixes(
        "noval20_mlp_a3_logit_decay",
        "noval20_mlp_a3_logit_no_entropy_decay",
    ),
    "A4": mean_curves_from_prefixes(
        "noval20_mlp_a4_entropy_decay",
        "noval20_mlp_a4_no_entropy_decay",
        seeds=(1, 2),
    ),
}

plot_run_mean_groups(
    mean_groups,
    split="test",
    smooth=50,
    title="Mean comparison by setup",
    ncols=2,
)